# 🛒 Superstore — Python & Pandas Data Exploration and Cleaning
**Objective:** Learn Python basics and perform data exploration and cleaning using Pandas  
**Dataset:** Sample - Superstore.csv (9,994 rows × 21 columns)  
**Steps:** Load → Explore → Handle Missing Values → Filter / Select → Remove Duplicates → Derive Columns → Save

---

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported')
print(f'  pandas : {pd.__version__}')
print(f'  numpy  : {np.__version__}')

Libraries imported
  pandas : 3.0.2
  numpy  : 2.4.4


---
## Step 1 · Load CSV into a Pandas DataFrame
Using `encoding='latin-1'` because the file contains special characters (e.g. ®, ™) that cause a UTF-8 decode error.

In [ ]:
df_raw = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')
print(f'Dataset loaded successfully!')
print(f'  Rows    : {df_raw.shape[0]:,}')
print(f'  Columns : {df_raw.shape[1]}')

Dataset loaded successfully!
  Rows    : 9,994
  Columns : 21


---
## Step 2 · Explore the Data
### 2-A  Shape, Columns & Data Types

In [ ]:
print('Shape:', df_raw.shape)
print('\nData Types:\n', df_raw.dtypes)

Shape: (9994, 21)

Data Types:
 Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64


### 2-B  First 5 Rows (`head`)

In [ ]:
df_raw.head()

 Row ID       Order ID Order Date  Ship Date      Ship Mode Customer ID   Customer Name   Segment       Country            City      State  Postal Code Region      Product ID        Category Sub-Category                                                Product Name    Sales  Quantity  Discount    Profit
      1 CA-2016-152156  11/8/2016 11/11/2016   Second Class    CG-12520     Claire Gute  Consumer United States       Henderson   Kentucky        42420  South FUR-BO-10001798       Furniture    Bookcases                           Bush Somerset Collection Bookcase 261.9600         2      0.00   41.9136
      2 CA-2016-152156  11/8/2016 11/11/2016   Second Class    CG-12520     Claire Gute  Consumer United States       Henderson   Kentucky        42420  South FUR-CH-10000454       Furniture       Chairs Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back 731.9400         3      0.00  219.5820
      3 CA-2016-138688  6/12/2016  6/16/2016   Second Class    DV-13045 Darrin Van Huff Cor

### 2-C  Last 5 Rows (`tail`)

In [ ]:
df_raw.tail()

 Row ID       Order ID Order Date Ship Date      Ship Mode Customer ID    Customer Name  Segment       Country        City      State  Postal Code Region      Product ID        Category Sub-Category                                                              Product Name   Sales  Quantity  Discount  Profit
   9990 CA-2014-110422  1/21/2014 1/23/2014   Second Class    TB-21400 Tom Boeckenhauer Consumer United States       Miami    Florida        33180  South FUR-FU-10001889       Furniture  Furnishings                                                    Ultra Door Pull Handle  25.248         3       0.2  4.1028
   9991 CA-2017-121258  2/26/2017  3/3/2017 Standard Class    DB-13060      Dave Brooks Consumer United States  Costa Mesa California        92627   West FUR-FU-10000747       Furniture  Furnishings                        Tenex B1-RE Series Chair Mats for Low Pile Carpets  91.960         2       0.0 15.6332
   9992 CA-2017-121258  2/26/2017  3/3/2017 Standard Class    DB-13060   

### 2-D  Statistical Summary (`describe`)

In [ ]:
df_raw[['Sales','Quantity','Discount','Profit']].describe().round(2)

              Sales     Quantity     Discount       Profit
count   9994.000000  9994.000000  9994.000000  9994.000000
mean     229.858001     3.789574     0.156203    28.656896
std      623.245101     2.225110     0.206452   234.260108
min        0.444000     1.000000     0.000000 -6599.978000
25%       17.280000     2.000000     0.000000     1.728750
50%       54.490000     3.000000     0.200000     8.666500
75%      209.940000     5.000000     0.200000    29.364000
max    22638.480000    14.000000     0.800000  8399.976000


> **Observations:**
> - `Sales` ranges $0.44–$22,638 — wide spread, a few very high-value items
> - `Discount` maximum is 0.8 (80%) — aggressive discounting exists
> - `Profit` minimum is negative — some orders are loss-making
> - `Quantity` max is 14 units per line item


---
## Step 3 · Handle Missing Values
### 3-A  Identify Missing Values

In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())
print(f'\nTotal missing cells: {df_raw.isnull().sum().sum()}')

Missing values per column:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0

Total missing cells: 0


✅ **No missing values found — the dataset is complete.**

In a real scenario with missing data you would use:
```python
# Fill numeric nulls with the median
df['Sales'].fillna(df['Sales'].median(), inplace=True)

# Fill categorical nulls with the mode
df['Region'].fillna(df['Region'].mode()[0], inplace=True)

# Drop rows where critical columns are null
df.dropna(subset=['Order ID', 'Customer ID'], inplace=True)
```

### 3-B  Check for Duplicate Rows

In [ ]:
print('Exact duplicate rows:', df_raw.duplicated().sum())

biz_mask = df_raw.duplicated(subset=['Order ID', 'Product ID'])
print('Business duplicates (Order ID + Product ID):', biz_mask.sum())
print()
print(df_raw[biz_mask][['Order ID','Product ID','Customer Name','Sales','Profit']].to_string(index=False))

Exact duplicate rows: 0
Business duplicates (Order ID + Product ID): 8

      Order ID      Product ID   Customer Name   Sales   Profit
CA-2016-129714 OFF-PA-10001970 Adam Bellavance  49.120  23.0864
US-2016-123750 TEC-AC-10004659      Ross Baird 291.960  54.7425
CA-2016-137043 FUR-FU-10003664    Logan Currie 286.380  83.0502
CA-2017-152912 OFF-ST-10003208      Brian Moss 544.380 157.8702
US-2014-150119 FUR-CH-10002965  Laurel Beltran 281.372 -12.0588
CA-2015-103135 OFF-BI-10000069 Shirley Schmidt  90.060  41.4276
CA-2017-118017 TEC-AC-10002006   Lena Cacioppo 102.336  14.0712
CA-2016-140571 OFF-PA-10001954   Sanjit Jacobs  45.680  21.0128


---
## Step 4 · Basic Operations — Filter Rows & Select Columns
### 4-A  Standardise Column Names, Parse Dates, Fix Postal Codes

In [ ]:
df = df_raw.copy()

# Parse date strings to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

# Postal Code: keep as zero-padded string
df['Postal Code'] = df['Postal Code'].astype(str).str.zfill(5)

# Standardise column names: lowercase with underscores
df.columns = [c.strip().lower().replace(' ','_').replace('-','_') for c in df.columns]

print('Cleaned column names:')
print(df.columns.tolist())
print(f'\norder_date dtype  : {df["order_date"].dtype}')
print(f'postal_code sample: {df["postal_code"].head(3).tolist()}')

Cleaned column names:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']

order_date dtype  : datetime64[ns]
postal_code sample: ['42420', '42420', '90036']


### 4-B  Filter — West Region & Technology Category

In [ ]:
west_tech = df[(df['region'] == 'West') & (df['category'] == 'Technology')]
print(f'Matching rows: {len(west_tech):,}')
print()
print(west_tech[['order_id','customer_name','product_name','sales','profit']].head(5).to_string(index=False))

Matching rows: 588

      order_id      customer_name                                                   product_name   sales  profit
CA-2014-115812    Brosina Hoffman                                 Mitel 5320 IP Phone VoIP phone 907.152 90.7152
CA-2014-115812    Brosina Hoffman                  Konftel 250 Conference phone - Charcoal black 911.424 68.3568
CA-2014-143336 Zuschuss Donatelli                                        Cisco SPA 501G IP Phone 213.480 16.0110
CA-2016-121755      Eric Hoffmann               Imation 8GB Mini TravelDrive USB 2.0 Flash Drive  90.570 11.7741
CA-2015-135545       Kunst Miller Verbatim 25 GB 6x Blu-ray Single Layer Recordable Disc, 3/Pack  13.980  6.1512


### 4-C  Filter — High-Value Orders (Sales > $1,000)

In [ ]:
high_value = df[df['sales'] > 1000]
print(f'High-value rows: {len(high_value):,}')
print()
print(high_value[['order_id','product_name','sales','profit']].head(5).to_string(index=False))

High-value rows: 468

      order_id                                                  product_name    sales     profit
CA-2014-115812                      Chromcraft Rectangular Conference Tables 1706.184    85.3092
CA-2015-106320                 Bretford CR4500 Series Slim Rectangular Table 1044.630   240.2649
US-2015-150630 Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish 3083.430 -1665.0522
CA-2016-117590                                                   GE 30524EE4 1097.544   123.4737
CA-2016-105816                              AT&T CL83451 4-Handset Telephone 1029.950   298.6855


### 4-D  Filter — Loss-Making Orders (Profit < 0)

In [ ]:
losses = df[df['profit'] < 0]
print(f'Loss-making rows  : {len(losses):,}')
print(f'Total loss amount : ${losses["profit"].sum():,.2f}')
print()
print(losses[['product_name','category','sales','discount','profit']].head(5).to_string(index=False))

Loss-making rows  : 1,871
Total loss amount : $-156,131.29

                                                                product_name        category     sales  discount     profit
                               Bretford CR4500 Series Slim Rectangular Table       Furniture  957.5775      0.45  -383.0310
Holmes Replacement Filter for HEPA Air Cleaner, Very Large Room, HEPA Filter Office Supplies   68.8100      0.80  -123.8580
                            Storex DuraTech Recycled Plastic Frosted Binders Office Supplies    2.5440      0.80    -3.8160
                                          Global Deluxe Stacking Chair, Gray       Furniture   71.3720      0.30    -1.0196
               Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish       Furniture 3083.4300      0.50 -1665.0522


### 4-E  Select Specific Columns

In [ ]:
core_cols = ['order_id','order_date','customer_name','segment',
             'region','category','sub_category','product_name',
             'sales','quantity','discount','profit']
df_core = df[core_cols]
print(f'Subset shape: {df_core.shape}')
print()
print(df_core.head(3).to_string(index=False))

Subset shape: (9994, 12)

      order_id order_date   customer_name   segment region        category sub_category                                                product_name  sales  quantity  discount   profit
CA-2016-152156 2016-11-08     Claire Gute  Consumer  South       Furniture    Bookcases                           Bush Somerset Collection Bookcase 261.96         2       0.0  41.9136
CA-2016-152156 2016-11-08     Claire Gute  Consumer  South       Furniture       Chairs Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back 731.94         3       0.0 219.5820
CA-2016-138688 2016-06-12 Darrin Van Huff Corporate   West Office Supplies       Labels   Self-Adhesive Address Labels for Typewriters by Universal  14.62         2       0.0   6.8714


---
## Step 5 · Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['order_id', 'product_id'])
after = len(df)

print(f'Rows before: {before:,}')
print(f'Rows after : {after:,}')
print(f'Removed    : {before - after}')
print('\nUsed business key [order_id, product_id] — keeps first occurrence')

Rows before: 9,994
Rows after : 9,986
Removed    : 8

Used business key [order_id, product_id] — keeps first occurrence


---
## Step 6 · Create Derived Columns

| New Column | Formula | Purpose |
|---|---|---|
| `total_amount` | `sales × quantity` | True revenue volume per line |
| `profit_flag` | sign of `profit` | Categorise P&L status |
| `discount_band` | bucketed `discount` | Group discount tiers |

In [ ]:
# total_amount = sales * quantity
df['total_amount'] = (df['sales'] * df['quantity']).round(2)

# profit_flag: categorise each row
df['profit_flag'] = df['profit'].apply(
    lambda x: 'Loss' if x < 0 else ('Break-Even' if x == 0 else 'Profit')
)

# discount_band: bucket discounts into tiers
df['discount_band'] = pd.cut(
    df['discount'],
    bins=[-0.001, 0, 0.10, 0.20, 0.30, 1.0],
    labels=['No Discount','1-10%','11-20%','21-30%','31%+']
)

print(f'Columns added — DataFrame now has {df.shape[1]} columns')
print()
print(df[['order_id','product_name','sales','quantity',
          'total_amount','profit_flag','discount_band']].head(8).to_string(index=False))

Columns added — DataFrame now has 24 columns

      order_id                                                     product_name    sales  quantity  total_amount profit_flag discount_band
CA-2016-152156                                Bush Somerset Collection Bookcase 261.9600         2        523.92      Profit   No Discount
CA-2016-152156      Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back 731.9400         3       2195.82      Profit   No Discount
CA-2016-138688        Self-Adhesive Address Labels for Typewriters by Universal  14.6200         2         29.24      Profit   No Discount
US-2015-108966                    Bretford CR4500 Series Slim Rectangular Table 957.5775         5       4787.89        Loss          31%+
US-2015-108966                                   Eldon Fold 'N Roll Cart System  22.3680         2         44.74      Profit        11-20%
CA-2014-115812 Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood  48.8600         7        342.02      Pr

In [ ]:
print('total_amount stats:')
print(df['total_amount'].describe().round(2))
print('\nprofit_flag counts:')
print(df['profit_flag'].value_counts())
print('\ndiscount_band counts:')
print(df['discount_band'].value_counts().sort_index())

total_amount stats:
count    9986.00
mean      696.28
std      1532.41
min         0.44
25%        93.06
50%       283.26
75%       719.62
max     44958.72
Name: total_amount, dtype: float64

profit_flag counts:
Profit        7821
Loss          2124
Break-Even      41
Name: profit_flag, dtype: int64

discount_band counts:
No Discount    4728
1-10%           131
11-20%          615
21-30%         1254
31%+           3258
Name: discount_band, dtype: int64


---
## Step 7 · Save the Cleaned Dataset as CSV

In [ ]:
df.to_csv('superstore_cleaned.csv', index=False)

print('Saved: superstore_cleaned.csv')
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print()
print('All columns in cleaned file:')
for col in df.columns:
    print(f'  {col}')

Saved: superstore_cleaned.csv
  Rows    : 9,986
  Columns : 24

All columns in cleaned file:
  row_id
  order_id
  order_date
  ship_date
  ship_mode
  customer_id
  customer_name
  segment
  country
  city
  state
  postal_code
  region
  product_id
  category
  sub_category
  product_name
  sales
  quantity
  discount
  profit
  total_amount
  profit_flag
  discount_band
